# Step 5: Ablation study of the main deep model

Each variant changes one thing and is **retrained from scratch** with the same seed, inputs, loop and checkpoint rule as the step 4 model (`full`). The attention news and user encoders are in every variant, so every variant still reads content and the user's history; what changes is the engineered evidence and the ID treatment.

| Variant | What changes | Question it answers |
|---|---|---|
| `no_candidate_attention` | without the candidate-aware attention branch (Group 10's user vector only) | Does letting the candidate pick the relevant history help? (Topic 7B) |
| `no_id_masking` | ID vectors never masked in training | Does forcing the network off ID look-ups help cold articles? (Topic 6) |
| `no_t1` | without the article's own behaviour | How much does the candidate's own history matter, cold articles included? |
| `no_t2` | without content-transferred behaviour | Does borrowed topic/entity behaviour add anything? |
| `no_t3` | without explicit content columns (codes, entity counts) | Do they add anything beyond the news encoder? |
| `no_u` | without the engineered user–article matching columns | Does explicit matching add to the attention user encoder? |
| `full_plus_smoothing` | adds content_smoothed_ctr (α = 20) | Does the interim report's hybrid feature help? |

**Running on Colab in pieces.** Each finished variant writes its prediction files and training history to Drive. Re-running the notebook skips variants that are already done, so the six runs can be spread across sessions or machines. Set `RUN_ONLY` to a subset to split the work.

> Metrics here come from `shared/eval_interim.py`. Final ablation tables and intervals come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [ ]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from shared import schema as S, selection as SEL
from shared.predictions import write_predictions, read_predictions
from shared.nrms_inputs import build_inputs
from shared import nrms_g10 as N, hybrid_model as H
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
CACHE_DIR = os.path.join(DATA_ROOT, "nrms_cache", DATA_KIND.lower())   # MiniLM / spaCy vectors from step 3b
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 4222

G10_CONFIG = dict(num_heads=2, dropout=0.1, user_heads=2, user_dropout=0.1)   # Group 10 Test_2, as step 3b
TRAIN = dict(epochs=10, lr=5e-4, batch_size=32, patience=3)                    # Group 10's loop + early stop
SELECT = "ndcg@10"           # checkpoint criterion: best validation nDCG@10
ALPHA = 20                   # content_smoothed_ctr strength for the +H variant (step 3a's validation choice)
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | device={DEVICE} | features: {FEATURE_ROOT}")
RUN_ONLY = os.environ.get("G11_DEEP_VARIANTS")          # e.g. "no_t1,no_t2" to split the runs; default: all
TO_RUN = [v for v in H.VARIANTS if v != "full"]
if RUN_ONLY:
    TO_RUN = [v for v in TO_RUN if v in RUN_ONLY.split(",")]
print("variants to run (if not already done):", TO_RUN)

## 2. Model inputs (same as step 4)

In [ ]:
t0 = time.time()
META_COLS = ["row_id", "impression_id", "news_id", "time", "label", "is_cold_10"]
meta = {s: pd.read_parquet(S.meta_path(FEATURE_ROOT, s), columns=META_COLS) for s in S.SPLITS}
inputs = build_inputs(RAW_DIR, meta, device=DEVICE, cache_dir=CACHE_DIR, seed=SEED)
for split in S.SPLITS:
    assert (inputs["impressions"]["impression_id"][inputs["splits"][split]["imp"]] == meta[split]["impression_id"].to_numpy()).all()
print(f"inputs built in {time.time() - t0:.0f}s; {inputs['info']['n_train_samples']:,} training samples "
      f"(1 click + 4 non-clicks each), {len(inputs['news']['title_abs']):,} articles")

## 3. Shared training function (same as step 4)

In [ ]:
def load_features(groups):
    """Scaled feature rows for every split, aligned with meta on row_id. Scaling is fitted on train only."""
    num_cols, cat_cols = H.variant_columns(groups)
    raw = {s: pd.read_parquet(S.features_path(FEATURE_ROOT, s), columns=["row_id"] + H.raw_columns(num_cols, cat_cols))
           for s in S.SPLITS}
    for s in S.SPLITS:
        assert np.array_equal(raw[s]["row_id"].to_numpy(), meta[s]["row_id"].to_numpy()), f"{s}: row_id misaligned"
    scaler = H.FeatureScaler(num_cols, cat_cols, alpha=ALPHA).fit(raw["train"])
    feats = {s: scaler.transform(raw[s]) for s in S.SPLITS}
    cat_sizes = [(int(max(raw[s][c].max() for s in S.SPLITS)) + 1, H.CAT_DIMS[c]) for c in cat_cols]
    return feats, num_cols, cat_cols, cat_sizes


def run_variant(name, score_splits=("val", "test"), log=print):
    """Train one variant from scratch, restore its best validation epoch, score the requested splits."""
    spec = H.VARIANTS[name]
    feats, num_cols, cat_cols, cat_sizes = load_features(spec["groups"])
    N.set_seed(SEED)
    model = H.HybridNRMS(inputs, len(num_cols), cat_sizes, id_mask_p=spec["id_mask_p"],
                         candidate_attention=spec.get("candidate_attention", True), **G10_CONFIG).to(DEVICE)
    val_meta, val_split = meta["val"], inputs["splits"]["val"]

    def val_fn(m):
        res = E.evaluate(val_meta, H.score_split(m, val_split, *feats["val"]))
        return {k: res[k] for k in ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10"]}

    t0 = time.time()
    history, best_epoch = H.train(model, inputs["train_samples"], *feats["train"], seed=SEED, val_fn=val_fn,
                                  select_metric=SELECT, log=log, **TRAIN)
    scores = {s: H.score_split(model, inputs["splits"][s], *feats[s]) for s in score_splits}
    return {"model": model, "history": history, "best_epoch": best_epoch, "scores": scores, "num_cols": num_cols,
            "cat_cols": cat_cols, "minutes": round((time.time() - t0) / 60, 1)}

## 4. Train each variant

Validation and test are scored for every variant. A variant whose prediction files already exist is skipped.

In [ ]:
def done(variant):
    return all(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__deep_{variant}__*__{s}.parquet"))
               for s in ("val", "test"))

import glob
for variant in TO_RUN:
    if done(variant):
        print(f"{variant}: already done, skipped")
        continue
    print(f"===== {variant} =====")
    r = run_variant(variant)
    cfg = f"{variant},id_mask={H.VARIANTS[variant]['id_mask_p']},epoch={r['best_epoch']}"
    for split, sc in r["scores"].items():
        write_predictions(meta[split]["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=f"deep_{variant}",
                          config_id=cfg, split=split, schema_version=S.SCHEMA_VERSION,
                          extra={"variant": variant, "best_epoch": r["best_epoch"], "selected_on": "val"})
    r["history"].to_csv(os.path.join(MODEL_DIR, f"deep_{variant}_history.csv"))
    print(f"{variant}: best epoch {r['best_epoch']}, {r['minutes']} min")
    del r

## 5. Results and change against the full model

The full model's predictions come from step 4. For nDCG@10 and Cold Recall@10 the 95% interval is a paired bootstrap over impressions (1,000 resamples).

In [ ]:
def load_scores(model_id, split):
    paths = glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__{model_id}__*__{split}.parquet"))
    if not paths:
        return None
    p, _ = read_predictions(sorted(paths)[-1])
    return meta[split][["row_id"]].merge(p, on="row_id", how="left")["predicted_score"].to_numpy()

rows, delta_rows = [], []
for split in ("val", "test"):
    base_sc = load_scores("deep_hybrid", split)
    assert base_sc is not None, "run step 4 first"
    base = E.per_impression_metrics(meta[split], base_sc)
    rows.append({"split": split, "variant": "full", **{k: v for k, v in E.summarise(base).items() if k in SHOW}})
    for variant in H.VARIANTS:
        if variant == "full":
            continue
        sc = load_scores(f"deep_{variant}", split)
        if sc is None:
            continue
        other = E.per_impression_metrics(meta[split], sc)
        rows.append({"split": split, "variant": variant, **{k: v for k, v in E.summarise(other).items() if k in SHOW}})
        row = {"split": split, "variant": variant}
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(base[metric], other[metric])
            row.update({f"Δ {metric}": bs["mean_diff"], f"{metric} CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
        delta_rows.append(row)
from IPython.display import display
display(pd.DataFrame(rows).set_index(["split", "variant"]).round(4))
pd.DataFrame(delta_rows).set_index(["split", "variant"]).round(4)

## 6. What the ablations show

*To be written from the results.*